todo: Generate a toy example of qvst, in the specific session, not the Hubbard session

In [1]:
import pennylane as qp
import pennylane as qml
import pennylane.estimator as qre
import numpy as np
import scipy
import matplotlib.pyplot as plt
import logging
from collections import defaultdict

from scipy.stats import rv_continuous
from tqdm import tqdm

logging.basicConfig(level=logging.INFO)

from sympy import (
    simplify,
    latex,
    symbols,
    init_printing,
    Matrix,
    eye,
    conjugate as conj,
    MatMul as mm,
    sqrt,
    Rational,
)
from sympy.physics.quantum import TensorProduct as tp, Ket

init_printing()

np.random.seed(4)
qp.numpy.random.seed(4)

# Helper

In [2]:
def scalarize(x):
    # extracta number from [1, 1] array because sympy
    return x[0, 0] if getattr(x, "shape", None) == (1, 1) else x


a = symbols("a1:17")
U = Matrix(4, 4, a)

# Creation / Annihilation

$N = a^\dagger a$

Creation op $a^\dagger$ adds one quantum of energy $\hbar\omega$. 

$a^\dagger=\sqrt{\frac{mω}{2\hbar}}\hat{x}-i\sqrt{\frac{1}{2m\omega\hbar}}\hat p$

# Pennylane Hubbard

This work is two fold. The seconnd is to enable coversion of operators to sparse matrix, which helps us calculate the spectral norm easier

Resource estimation 
- What to show: derive a standard Trotter bound, explain why it is often loose, and produce a more realistic estimate for a specific Hamiltonian instance using commutator structure and problem parameters.
- For a usable latice $10 \times 10 \times 10$, it is impposible to run and estimate the resource
- Resource estimation:
    - “Is this algorithm ever useful on physically realizable hardware, or is it a mathematical object that only runs in a physicist's imagination?”
    - Wall time before decohesion, circuit depth?
    - How many qubits do Trotter still make sense
    - How many gates / qubits to solve a problem of this size 
    - What is the error, as the function of resource (https://pennylane.ai/qml/demos/tutorial_estimator_hamiltonian_simulation_gqsp#heisenberg-model-for-nmr-spectral-prediction)
    - Compare to classic?

- Inspiration / reference:
    - https://arxiv.org/pdf/2603.05479
- Comparison method: We exact $\ket{\psi_t}=\exp(−iHt\ket{\psi_0})$. Then compare the fidelity $F(\rho ,\sigma)=|\braket{\psi_\rho|\psi_\sigma}|^2$
    - https://dojo.qulacs.org/en/latest/notebooks/4.2_trotter_decomposition.html. Seems to have interesting comparison method  





Error in most simulation methods typically scales like:

__todo__ verify
- Trotter: Grows with both step size and total time often roughly: error∼O(t⋅Δtk)


**Plots**



Even more important in practice:


Then compare:

“Who gets lower error for the same cost?”

That’s the real quantum algorithm comparison.

3. Multiple random states + multiple times

Increasing t is not just a trick—it reveals:

phase accumulation errors
non-commutativity errors (Trotter-specific)
instability in approximations

So it’s standard practice to test:

short-time regime (sanity)
intermediate regime (comparison)
long-time regime (robustness)

## Failure modes
#### Failure mode 1: The work was solid, but not decision-changing
- Symptoms: Doesn’t beat candidates with stronger direct research records.
A supporting signal, a primary hiring signal.
- Treatment: 
publication?
stronger mathematical depth?
clearer fit to team?

#### Failure mode 2: Question choosing
- Symptoms
You optimized for breadth, coverage instead of decision-grade insight.
Why this problem, why now, why this comparison, what you deliberately excluded, what the real question is.

- Treatment:
Instead of “I compared several approaches”, say “I investigated which assumptions dominate the resource estimate in X setting, and found Y”
Write the
“Problem selection rationale”
“Why this question matters”
“Scope decisions and excluded alternatives”



#### Failure mode 3: The mentor was helpful, but not actually a sponsor
Expected

#### Failure mode 4: You became better, but the market still preferred more direct candidates
This is the harshest but also most normal outcome.

That means your project may have improved your profile without making you top-ranked.
What to do next
Option A — Change the position
Option B — Change the bar you’re trying to clear


#### Failure mode 6: You over-focused on the artifact and under-invested in market spread
You build the thing. But you didn’t build enough parallel opportunities.

What that means
You treated the artifact as if it would automatically create demand.




## Anti fragile
Create that remains useful, regardless of the results. Then build the missing experience or target a different entry point.

a target list of companies/roles,
2–3 tailored positioning variants,
people to send it to,
a short message template,
one portfolio page or concise summary

#### Cut lost triggers
one question, one baseline, one comparison, one expected insight

##### Trigger 2 — Novelty mirage
If by midpoint you still can’t identify: a sharp claim, a useful result, or a specific technical lesson,

##### Trigger 3 — Sponsorship failure
If after producing something concrete, no one knowledgeable is willing to say: “this is solid,”

##### Trigger 4 — Market mismatch
If the feedback is repeatedly: “good work, but we need deeper direct experience”

#### Diagnosis
What exactly blocked conversion into opportunity?

not enough direct research?
wrong topic?
insufficient sharpness?
wrong role?
too much competition?
not enough sponsorship?

Only then choose the next move.
1. Reclassify yourself honestly
Which are you now?

still mostly VQE?
broader quantum algorithms candidate?
tooling/benchmarking candidate?
research engineer more than researcher?
better suited to adjacent roles?

4. Choose a different lever
5. Move with a new strategy, not the same one louder

## Hyperparams

In [3]:
n_cells = [2, 2, 2]
t = 1.0
U = 4.0

num_state = 10  # size of dataset, number of states we need

## Get the Hamiltonian

In [4]:
H = qp.spin.fermi_hubbard(
    lattice="cubic",
    n_cells=n_cells,
    hopping=t,
    coulomb=U,
    boundary_condition=False,
    mapping="jordan_wigner",
)
n_qubits = H.num_wires

## Prepare dataset
### State preparation

Neel state ($\ket{0101...01}$) is the standard state for studying ferromagnetics problem


- https://pennylane.ai/qml/demos/tutorial_haar_measure/
- https://pennylane.ai/qml/demos/tutorial_unitary_designs/

Every rotation or op in SU(2) can be decomposed like at https://arxiv.org/pdf/1708.00735

Harr about measurement and it helps sample uniformly at random

Groundtruth generation: `qml.evolve()` would take a long time, so we rely on scipy sparse matrix exponential, which is relevant for Hubbard model. Then we use the $\braket{\psi|\mathcal{M}|\psi}$ to calculate the expected value of measurement $\mathcal{M}$. We are going to exploit the sparseness of Hubbard

In [5]:
def measure_fidelity(a: np.matrix, b: np.matrix):
    column_wise_dp = np.einsum("ij,ij->j", a, b)
    return np.abs(column_wise_dp) ** 2

## Trotterization
I set the state as the eigenstate from H. This way we can compare the accuracy of different methods

In [6]:
def prepare_neel_state(wires):
    """Néel state: alternating up/down on bipartite lattice"""
    for site in range(wires // 2):
        if site % 2 == 0:
            qml.PauliX(wires=2 * site)
        else:
            qml.PauliX(wires=2 * site + 1)


def generate_groundtruth(H, psi, time):
    matrix_H = H.sparse_matrix()
    return scipy.sparse.linalg.expm_multiply(-1j * matrix_H * time, psi)

In [7]:
dev = qml.device("lightning.qubit", wires=n_qubits)


def prepare_neel_state_array(wires):
    """
    Behave like
    ```
      prepare_neel_state(wires)
      qp.state()
    ```
    without having to setup the whole circuit
    Explain why this is false
    """
    n_sites = wires // 2
    index = 0
    for site in range(n_sites):
        if site % 2 == 0:
            qubit = 2 * site  # spin-up
        else:
            qubit = 2 * site + 1  # spin-down
        index += 2 ** (wires - 1 - qubit)  # PennyLane MSB-first convention
    state = np.zeros(2**wires)
    state[index] = 1.0
    return state


def double_occupancy_observable(wires, return_matrix=False):
    """
    ⟨Z↑ Z↓⟩ on a site gives ⟨n↑ n↓⟩ (double occupancy)
    after a trivial linear shift:
    ⟨n↑ n↓⟩ = (1 - ⟨Z↑⟩ - ⟨Z↓⟩ + ⟨Z↑Z↓⟩)/4
    """
    obs = 0
    site_list = range(wires // 2)

    for site in site_list:
        up = 2 * site  # qubit for spin up
        down = 2 * site + 1  # qubit for spin down
        obs += qml.PauliZ(up) @ qml.PauliZ(down)
    if return_matrix:
        return (obs / len(site_list)).sparse_matrix()
    else:
        return obs / len(site_list)


@qml.qnode(dev, interface=None)
def target_circuit(hamiltonian, time: int, wires: int, steps=10, order=1):
    prepare_neel_state(wires)
    qml.TrotterProduct(hamiltonian, n=steps, time=time, order=order)
    return qml.expval(double_occupancy_observable(wires))


def groundtruth_circuit(hamiltonian, time: int, wires: int):
    """
    Simluate exact evolution
    """
    initial_state = prepare_neel_state_array(wires)
    final_state = generate_groundtruth(hamiltonian, initial_state, time)
    ket_final = scipy.sparse.csc_array(final_state[:, np.newaxis])
    bra_final = scipy.sparse.csr_array(final_state[np.newaxis, :].conj())
    return (
        (bra_final @ double_occupancy_observable(wires, return_matrix=True) @ ket_final)
        .todense()
        .flatten()[0]
    )

In [8]:
# ## plot logic
# t_values = np.linspace(0, 2, 75)

# target_series = []
# groundtruth_series = []

# for t in tqdm(t_values):
#     target_val = target_circuit(H, t, n_qubits)
#     groundtruth_val = groundtruth_circuit(H, t, n_qubits)

#     target_series.append(target_val)
#     groundtruth_series.append(groundtruth_val)

# # Plot
# plt.figure(figsize=(8, 5))

# plt.plot(t_values, target_series, label="Trotter")
# plt.plot(t_values, groundtruth_series, label="Groundtruth")

# plt.xlabel("t")
# plt.ylabel("Circuit Output")
# plt.title("Target vs Groundtruth Circuit")
# plt.legend()
# plt.grid(True)

# plt.show()

### Resource estimation

https://pennylane.ai/qml/demos/re_how_to_use_pennylane_for_resource_estimation#fast-estimation-with-less-information

According to [IBM blog](https://www.ibm.com/quantum/blog/qdc-2024), Heron is capable of 5000 gates before decoherence. To simulate the situation on a real quantum machine, let's say we have a 50000 gates machine, and optimize our algorithm around this constraints

In [9]:
%%time
resources_exec = qre.estimate(target_circuit)(H, 0.1, n_qubits, order=1)
print(resources_exec)

--- Resources: ---
 Total wires: 16
   algorithmic wires: 16
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 3.969E+4
   'T': 3.168E+4,
   'CNOT': 4.640E+3,
   'X': 8,
   'Z': 480,
   'S': 960,
   'Hadamard': 1.920E+3
CPU times: user 34.2 ms, sys: 1.99 ms, total: 36.2 ms
Wall time: 35.1 ms


In [10]:
%%time
resources_exec = qre.estimate(target_circuit)(H, 0.1, n_qubits, order=2)
print(resources_exec)

--- Resources: ---
 Total wires: 16
   algorithmic wires: 16
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 7.839E+4
   'T': 6.252E+4,
   'CNOT': 9.224E+3,
   'X': 8,
   'Z': 942,
   'S': 1.884E+3,
   'Hadamard': 3.804E+3
CPU times: user 30.6 ms, sys: 1.61 ms, total: 32.3 ms
Wall time: 31.3 ms


Because the Hamiltonian is small, we don't have to go to the symbolic route, which is counting the appearances of Paulis and using `qre.PauliHamiltonian` and `qre.TrotterPauli`

Let's see the error using the link from https://pennylane.ai/qml/demos/tutorial_error_prop.
There are several ways to tighten the bound
1. As of Childs et al the error of Trotter is not 1-norm but spectral-norm
2. More intelligent fragmentation, while reordering fragments can be a big error reduceing Intuitively, given $H=A+B+C$
3. When to increase step in Trotter, and when to increases order?
    1. With a quantum machine with 100 Mhz, this depth etc
4. Rather than lowering the `t` of Trotter, smart fragmentation and ordering frequently give bigger practical gains than going to order 4 or 6. Pennylane has `labs` with fragmentation methods

### Error estimation

It is not easy enough to deal with big H in pennylane, because sparse matrix is not fully utilized

In [11]:
qml.operation.Operator.sparse_matrix(qml.TrotterProduct(H, n=10, time=1.5, order=1))

SparseMatrixUndefinedError: 

In [ ]:
H

In [ ]:
qml.operation.Operator.sparse_matrix(
    qml.TrotterProduct(qml.X(0) + qml.Y(1), n=10, time=1.5, order=1)
)

In [ ]:
qml.matrix(
    qml.TrotterProduct(
        qml.Y(0)
        + qml.Z(1)
        + qml.Z(2)
        + qml.Z(3)
        + qml.Z(4)
        + qml.Z(5)
        + qml.Z(6)
        + qml.Z(7)
        + qml.Y(8),
        n=10,
        time=1.5,
        order=1,
    )
)

In [ ]:
from pennylane.resource import SpectralNormError

SpectralNormError.get_error(op, op)

In [ ]:
m1 = qml.matrix(exact_op, wire_order=wire_order)
m2 = qml.matrix(approximate_op, wire_order=wire_order)
return qml.math.max(qml.math.svd(m1 - m2, compute_uv=False))

error = SpectralNormError.get_error(exact_op, approx_op)  # Expensive to compute
print(f"Error from Suzuki-Trotter algorithm: {error:.5f}")

Seems good, can we do with a bigger scale?

In [ ]:
%%time
H_big = qp.spin.fermi_hubbard(
    lattice="cubic",
    n_cells=[10, 10, 10],
    hopping=1,
    coulomb=U,
    boundary_condition=False,
    mapping="jordan_wigner",
)
H_big.num_wires

In [ ]:
%%time
# resources_exec = qre.estimate(target_circuit)(H_big, times[0], n_qubits)
# CPU times: user 22min 16s, sys: 377 ms, total: 22min 17s
# Wall time: 22min 17s

In [ ]:
print(resources_exec)
"""
--- Resources: ---
 Total wires: 2000
   algorithmic wires: 2000
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 4.625E+8
   'T': 1.272E+8,
   'CNOT': 3.202E+8,
   'Z': 2.160E+6,
   'S': 4.320E+6,
   'Hadamard': 8.640E+6
"""

Looks like we can't do it without costing a lot of time. The next solution is to create a symbolic circuit to help

In [ ]:
def convert_hamiltonian_to_pauli_dictionary(
    hamiltonian: qp.ops,
) -> list[dict[str, int], float]:
    """
    People can read from the Hamiltonian, but we have the particular H in this example, we can use that directly
    return something like
    pauli_dictionary = {
        "XX": ...,
        "YY": ...,
        "ZZ": ...
    }
    """
    pauli_dict = defaultdict(int)
    one_norm = 0
    for operand in hamiltonian.operands:
        term, gate = operand.terms()
        assert len(term) == len(gate) == 1
        term = term[0]
        gate = gate[0]
        if isinstance(gate, qp.operation.Operation):  # single gate ...
            if not isinstance(gate, qp.ops.identity.Identity):
                pauli_dict[gate.label()] += 1
            continue
        one_norm += term
        for gate_op in gate.operands:  # ... X @ Y @ qml.Z @ etc
            pauli_dict[gate_op.label()] += 1
    return pauli_dict, one_norm

In [ ]:
pauli_dict, norm = convert_hamiltonian_to_pauli_dictionary(H)

In [ ]:
symbolic_hamiltonian = qre.PauliHamiltonian(
    num_qubits=len(H.wires),
    pauli_terms=pauli_dict,
)


def symbolic_circuit(hamiltonian, num_steps, order, num_qubit):
    qre.UniformStatePrep(num_states=2**num_qubit)
    qre.TrotterPauli(hamiltonian, num_steps, order)

In [ ]:
print(qre.estimate(symbolic_circuit)(symbolic_hamiltonian, 100, 1, n_qubits))

A wild difference between
```
--- Resources: ---
 Total wires: 16
   algorithmic wires: 16
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 6.681E+6
   'T': 6.392E+6,
   'CNOT': 2.233E+5,
   'Z': 9.402E+3,
   'S': 1.880E+4,
   'Hadamard': 3.800E+4
```
and 
```
--- Resources: ---
 Total wires: 16
   algorithmic wires: 16
   allocated wires: 0
     zero state: 0
     any state: 0
 Total gates : 1.338E+6
   'T': 1.338E+6,
   'Hadamard': 16
```
   

In [ ]:
print(qre.estimate(symbolic_circuit)(symbolic_hamiltonian, 100, 1, n_qubits))

## QSVT

In [ ]:
qp.BlockEncode(matrix_H, n_qubits * 2)

In [ ]:
@qml.qnode(dev, interface=None)
def target_circuit(state: np.ndarray, time: int, wires: int):
    qml.StatePrep(state, wires=range(wires))

    for phi in phases:
        qml.RZ(phi, wires=0)
    qml.BlockEncode(coeffs, ops, wires=[0, 1])

    qml.QSVT(
        block_encoding, phases=compute_qsvt_phases(time, H_norm=1.0), wires=range(wires)
    )

    return qml.state()

In [ ]:
@qml.qnode(dev, interface=None)
def target_circuit(state: np.ndarray, time: int, wires: int):
    qml.StatePrep(state, range(wires))
    qml.TrotterProduct(H, time=time, n=100, order=2)
    return qml.state()

# Test pennylane resource est

In [ ]:
# n_cell = 100
n_cell = 25
num_steps = 10
order = 6


def pauli_quantities(n_cell):
    n_q = 2 * n_cell**2
    n_xx = n_cell**2
    n_yy = n_cell * (n_cell - 1)
    n_zz = n_yy
    return n_q, n_xx, n_yy, n_zz


n_q, n_xx, n_yy, n_zz = pauli_quantities(n_cell)
pauli_word_distribution = {"XX": n_xx, "YY": n_yy, "ZZ": n_zz}

kitaev_H = qre.PauliHamiltonian(
    num_qubits=n_q,
    pauli_terms=pauli_word_distribution,
)


def circuit(hamiltonian, num_steps, order):
    qre.UniformStatePrep(
        num_states=2**n_q
    )  # uniform superposition over all basis states
    qre.TrotterPauli(hamiltonian, num_steps, order)

In [ ]:
%%time
res = qre.estimate(circuit)(kitaev_H, num_steps, order)
print(res)

# ✔️ Create a 3D lattice

In [ ]:
# plot the lattice


def plot(lattice, figsize=None, showlabel=True):

    nodes = lattice.lattice_points
    fig = plt.figure()

    ax = fig.add_subplot(111, projection="3d")

    for edge in lattice.edges:
        start_index, end_index, color = edge
        start_pos, end_pos = nodes[start_index], nodes[end_index]

        x_axis = [start_pos[0], end_pos[0]]
        y_axis = [start_pos[1], end_pos[1]]
        z_axis = [start_pos[2], end_pos[2]]
        ax.plot(x_axis, y_axis, z_axis, color="gold")

        if showlabel:
            for index, pos in enumerate(nodes):
                ax.text(
                    pos[0] - 0.02,
                    pos[1] + 0.01,
                    pos[2] + 0.01,
                    str(index),
                    color="gray",
                )
    ax.scatter(nodes[:, 0], nodes[:, 1], nodes[:, 2], color="dodgerblue")

    ax.axis("off")
    plt.show()


lattice = qml.spin.generate_lattice("cubic", n_cells)
plot(lattice)

# Measurement

In [ ]:
dev = qp.device("default.qubit", wires=1, shots=1000)


@qp.qnode(dev)
def circuit():
    qp.H(0)
    return qp.expval(qp.PauliZ(0)), qp.sample(qp.PauliZ(0))


exp_val, samples = circuit()
print(exp_val)

# Convert measurement outcomes (+1, -1) to probabilities
p_plus = np.mean(samples == 1.0)
p_minus = np.mean(samples == -1.0)


print(f"P(E = +1) ≈ {p_plus}")
print(f"P(E = -1) ≈ {p_minus}")

# --- Theoretical Born rule ---
# State after H|0>:
# |ψ> = 1/sqrt2|0> + 1/sqrt(2)|1>
# <0|H|0> =

# p_plus_theory = np.cos(theta/2)**2
# p_minus_theory = np.sin(theta/2)**2

# print("\nBorn rule prediction:")
# print(f"P(E = +1) = {p_plus_theory}")
# print(f"P(E = -1) = {p_minus_theory}")

In [ ]:
samples == 1.0

In [ ]:
np.linalg.eig(1 / np.sqrt(2) * np.array([[1, 1], [1, -1]]))

# ✔️ Sanity check on Trotter

In [ ]:
hada = 1 / np.sqrt(2) * np.array([[1, 1], [1, -1]], dtype=float)
u = scipy.linalg.expm(-1j * hada * 100)
state = np.array([[1], [0]])
np.matrix(u @ state).H @ hada @ (u @ state)

In [ ]:
1 / np.sqrt(2) * qml.matrix(qml.X(0) + qml.Z(0)) - qml.matrix(qml.H(0))

In [ ]:
import pennylane as qml

dev = qml.device("lightning.qubit", wires=1)


@qml.qnode(dev, interface=None)
def circuit():
    # very big error when time == nsteps. nsteps >> time
    # Hadamard in disguise
    coeffs = [1 / np.sqrt(2), 1 / np.sqrt(2)]
    ops = [qml.X(0), qml.Z(0)]
    H = qml.dot(coeffs, ops)
    qml.TrotterProduct(H, time=10, n=10 * 100, order=4)
    return qml.expval(H)


circuit()

# QSVT

Basic QSP/QSVT works best with polynomials of definite parity (even or odd) that stay bounded on [-1,1]. ??

What do we do? We implement a primitive QSVT
- So if you can approximate the exponential with a polynomial: $e^{−iHt}≈P(H)$ then QSVT gives you a circuit for time evolution.
- In practice, people approximate the exponential using things like: Chebyshev polynomials or Jacobi–Anger expansion

<div class="alert alert-block alert-warning">Question</div>

- Projection $\stackrel{?}{=}$ step function $\stackrel{?}{=}$ walk operator.
- QVST engineers a circuit so that interference reproduces exactly this polynomial action on eigenvalues.


After a bunch of hurried deep dive and AI Slop explanation, here we go again

In [ ]:
Z = Matrix([[1, 0], [0, -1]])

X = Matrix([[0, 1], [1, 0]])

# Define H = 1/2 (Z + X)
H_QVST = Rational(1, 2) * (Z + X)
eigs = H_QVST.eigenvects()

In [ ]:
# reconstruct H from eigenvectors
eig_vec_1 = eigs[0][2][0] / eigs[0][2][0].norm()
eig_vec_2 = eigs[1][2][0] / eigs[1][2][0].norm()
component_1 = eigs[0][0] * (eig_vec_1 * eig_vec_1.H)
component_2 = eigs[1][0] * (eig_vec_2 * eig_vec_2.H)
simplify(component_1 + component_2)

Now choose a $P(x), x \in \mathbb{R}$. Then $P(H)=\sum_j P(\lambda)\bra {v_j} \ket{v_j}$. They make this clever claim that we can do P(H) just by applying P to the eigenvalues. And the domain of $P$ is now suddenly also $\mathbb{R}^{n \times n}$, $n$ goes to infinity. Talk about sloppy notation.

What are eigenvalues again?

In [ ]:
logging.info(f"Eigenvalues {eigs[0][0], eigs[1][0]}")

So they make this clever claim that we can do $P(H)$ just by applying $P$ to the eigenvalues. Now let's say $P(H) = H^2$

In [ ]:
H_QVST * H_QVST.H

In [ ]:
P_component_1 = eigs[0][0] ** 2 * (eig_vec_1 * eig_vec_1.H)
P_component_2 = eigs[1][0] ** 2 * (eig_vec_2 * eig_vec_2.H)
simplify(P_component_1 + P_component_2)

<div class="alert alert-block alert-info">They ain't shitting us bro</div>

```
<div class="alert alert-block alert-warning">They ain't shitting us bro</div>

<div class="alert alert-block alert-success">They ain't shitting us bro</div>

<div class="alert alert-block alert-danger">Yo<div>
```

### Build a quantum circuit to Block Encode
This is the only place where I intend to use qp lib, because we do it manually first

In [ ]:
H_QVST

In [ ]:
encoded = qml.matrix(qml.BlockEncode(np.array(H_QVST, dtype=float), wires=[0, 1]))

In [ ]:
# Define device (2 qubits: 0: ancilla, 1: system)
dev_qsvt = qml.device("lightning.qubit", wires=2)


@qml.qnode(dev_qsvt)
def circuit_be():
    qml.BlockEncode(np.matrix(H_QVST, dtype=float), wires=[0, 1])
    return qml.state()


circuit_be()

<div class="alert alert-block alert-warning">A correct block encoding U should satisfy: $⟨0∣U∣0⟩=H/α$</div>

Then $⟨0_a∣U∣0_a⟩$ means: “Take the top-left block.”



In [ ]:
state_0 = np.array([[1, 0, 0, 0]])
state_0 @ encoded @ state_0.T

In https://pennylane.ai/qml/demos/tutorial_intro_qsvt they say

> We now ask the crucial question that will get everything started: what happens if we repeatedly alternate multiplication of this matrix by some other matrix? 🤔 There are multiple choices for the “other matrix"

What is alternate multiplication. Why do we even care about it

Interesting to read for quantum chem knowledge: https://pennylane.ai/blog/2022/09/how-to-use-the-hartree-fock-method-in-pennylane

In [ ]:
c2h6 = qp.data.load("qchem", molname="C2H6", bondlength=1.535, basis="STO-3G")[0]

In [ ]:
H = c2h6.hamiltonian

$H$ has 32 wires. If we are to calculate the GS energy, we do a double excitation gate, with params, and try to gradient descend to minimum.

Now, without back props and eigenvalues finding, how do I find GS energy. It is smallest eigenvalue of $H$

In [ ]:
len(H.terms()[0])

# LCU
I am trying to implement LCU in sympy. The $LCU=2I+X$, $\psi=\alpha|0\rangle+\beta|1\rangle$. Now the result is incorrect. What is wrong. Code is

## Theoretical proof

$U_{ij}=<j|U|i>$

In [ ]:
U

In [ ]:
# let i=0,1 and j=0,1,2. We want U[0..1, 0..2] (2*3 matrix)
U_i = U * Matrix([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0]]).T
U_ij = Matrix([[1, 0], [0, 1], [0, 0], [0, 0]]).T * U_i
U_ij

## State

In [ ]:
alpha, beta = symbols("alpha beta", complex=True)
ket0 = Matrix([1, 0])
ket1 = Matrix([0, 1])
ket00 = Matrix([1, 0, 0, 0])

# First qubit: alpha|0> + beta|1>
psi = Matrix([alpha, beta])
state = tp(Matrix([1, 0]), psi)

print("State")
state

## Operator

In [ ]:
X = Matrix([[0, 1], [1, 0]])

XX = tp(X, X)
II = eye(4)
I = eye(2)
LCU = 2 * I + X
# PREP
a = [2, 1]
lmbda = sum(a)
prep_0 = sqrt(Rational(a[0], lmbda)) * ket0 + sqrt(Rational(a[1], lmbda)) * ket1
# SELECT op
# select = ket0*(ket0.H*I) + ket1*(ket1.H*X)
# simplify(scalarize(prep_full.H*(select_0 + select_1))*prep_full)
select = tp(ket0 * ket0.H, I) + tp(ket1 * ket1.H, X)  # |k>|U_k|psi>
tp(prep_0.H, I) * select * tp(prep_0, I) * psi

The definition of `SELECT` operator is $|k\rangle U_k\psi$ (https://pennylane.ai/qml/demos/tutorial_lcu_blockencoding), why is it implemented as a tensor product, nor normalt matmul

### Step

In [ ]:
(prep_0.shape, Matrix([prep_0[1], -prep_0[0]]).shape)

In [ ]:
# prep = [prep_0, Matrix([prep_0[1], -prep_0[0]])]
prep = Matrix([[prep_0[0], prep_0[1]], [prep_0[1], -prep_0[0]]])

prep

In [ ]:
be = tp(prep, I) * select * tp(prep, I)

In [ ]:
be, be * be.H

In [ ]:
2 * I + X

In [ ]:
from sympy import Matrix, eye

# Ancilla basis
ket0 = Matrix([1, 0])
ket1 = Matrix([0, 1])

# Target operators
I2 = eye(2)
X = Matrix([[0, 1], [1, 0]])

# SELECT operator (4x4)
SELECT = tp(ket0 * ket0.T, I2) + tp(ket1 * ket1.T, X)

PREP = sqrt(Rational(2, 3)) * ket0 + sqrt(Rational(1, 3)) * ket1
LCU_op = tp(PREP.H, eye(2)) * SELECT * tp(PREP, eye(2))  # 2x2 operator on target
final_state = simplify(LCU_op * psi)
final_state

## Applying to real case

Given linear combination of two unitaries: $KU+V$
Show that if we measure $|0>$ on the auxiliary wire, the state on the main register is $(KU+V)|\psi>$ up to normalization.

Attempt:
With $M$ as main register and $A$ as auscilliary register

$<0_AI_M|0_AI_M>KU+V|0_A\psi_M>=<0_AI_M|0_AI_M>|0_A>KU+V|0_A\psi_M>$
Now how to reduce that?

1. Prep: $\frac{1}{}$

# ✔️ Deriving Controls gate

### CNOT
$|0><0|I + |1><1|X$

In [ ]:
X

In [ ]:
tp(ket0 * ket0.H, I) + tp(ket1 * ket1.H, X)

In [ ]:
ket0 * ket0.H + ket1 * ket1.H

In [ ]:
ket0.H * ket1

## CCNOT (Toffoli)
Only inverse if 1st and 2nd bit are 1

$|0><0|I + |1><1|X$

In [ ]:
ket00 = tp(ket0, ket0)
ket01 = tp(ket0, ket1)
ket10 = tp(ket1, ket0)
ket11 = tp(ket1, ket1)

In [ ]:
toffoli = (
    tp(ket00 * ket00.H, I)
    + tp(ket01 * ket01.H, I)
    + tp(ket10 * ket10.H, I)
    + tp(ket11 * ket11.H, X)
)

In [ ]:
toffoli

In [ ]:
tp(state, ket0)

In [ ]:
toffoli * tp(state, ket0)

# Spectral theorem vs SVD

In [ ]:
A = np.array([[4, 1], [1, 3]], dtype=float)


eigenvalues, Q = np.linalg.eigh(A)

Lambda = np.diag(eigenvalues)

print("\nEigenvalues:")
print(eigenvalues)

print("\nOrthogonal eigenvector matrix Q:")
print(Q)

print("\nDiagonal matrix Λ:")
print(Lambda)

# Reconstruct A
A_reconstructed = Q @ Lambda @ Q.T

print("\nQ Λ Q^T:")
print(A_reconstructed)

print("\nCheck spectral decomposition:")
print(np.allclose(A, A_reconstructed))

In [ ]:
# =========================================================
# 2. SINGULAR VALUE DECOMPOSITION
# =========================================================
# A = U Σ V^T

U, singular_values, VT = np.linalg.svd(A)

Sigma = np.zeros_like(A)
np.fill_diagonal(Sigma, singular_values)

print("\nU:")
print(U)

print("\nSingular values:")
print(singular_values)

print("\nΣ:")
print(Sigma)

print("\nV^T:")
print(VT)

# Reconstruct A
A_svd = U @ Sigma @ VT

print("\nU Σ V^T:")
print(A_svd)

print("\nCheck SVD reconstruction:")
print(np.allclose(A, A_svd))

# =========================================================
# 3. DIAGONALIZATION ILLUSTRATION
# =========================================================

# Spectral diagonalization:
D1 = Q.T @ A @ Q

print("\nQ^T A Q =")
print(D1)

# SVD diagonalization-like form:
D2 = U.T @ A @ VT.T

print("\nU^T A V =")
print(D2)

In [ ]:
#!pip install pennylane --target=/kaggle/working/